In [13]:
import os
import re
import cv2
import insightface

import numpy as np
from insightface.app import FaceAnalysis

In [14]:
# app = FaceAnalysis(name="buffalo_l")
# app = FaceAnalysis(name="buffalo_s")
app = FaceAnalysis(name="buffalo_sc")
app.prepare(ctx_id=-1)  # CPU

# app = FaceAnalysis(name="buffalo_l")
# app.prepare(ctx_id=0)  # GPU

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\muysengly/.insightface\models\buffalo_sc\det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\muysengly/.insightface\models\buffalo_sc\w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [15]:
def get_face_embedding(image_path):
    """Extract face embedding from an image"""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    faces = app.get(img)

    if len(faces) < 1:
        raise ValueError("No faces detected in the image")
    if len(faces) > 1:
        print("Warning: Multiple faces detected. Using first detected face")

    return faces[0].embedding


def compare_faces_cosine(emb1, emb2, threshold=0.65):
    """Compare two embeddings using cosine similarity"""
    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity, similarity > threshold


def list_folders(directory, pattern=None):
    if pattern is not None:
        return [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f)) and re.search(pattern, f)]
    else:
        return [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]


def list_files(directory, pattern=None):
    if pattern is None:
        return [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    else:
        return [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f)) and re.search(pattern, f)]

In [16]:
datas_dir_file = []
dirs = list_folders("./data")
for dir in dirs:
    files = list_files(f"./data/{dir}/")
    datas_dir_file.append([dir, files])


datas_dir_file

[['angelina_jolie', ['1.jpg']],
 ['bradley_cooper', ['1.jpg']],
 ['kate_siegel', ['1.jpg']],
 ['MUY Sengly',
  ['001.jpg',
   '002.jpg',
   '003.jpg',
   'WIN_20250310_13_11_34_Pro.jpg',
   'WIN_20250310_13_11_36_Pro.jpg',
   'WIN_20250310_18_20_32_Pro.jpg',
   'WIN_20250310_18_20_34_Pro.jpg',
   'WIN_20250310_18_20_35_Pro.jpg',
   'WIN_20250310_18_20_37_Pro.jpg',
   'WIN_20250310_18_20_39_Pro.jpg',
   'WIN_20250310_18_20_40_Pro.jpg',
   'WIN_20250310_18_20_41_Pro.jpg',
   'WIN_20250310_18_20_42_Pro.jpg',
   'WIN_20250310_18_20_43_Pro.jpg',
   'WIN_20250310_18_20_44_Pro.jpg',
   'WIN_20250310_18_20_45_Pro (2).jpg',
   'WIN_20250310_18_20_45_Pro.jpg',
   'WIN_20250310_18_20_46_Pro.jpg',
   'WIN_20250310_18_20_48_Pro.jpg',
   'WIN_20250310_18_20_50_Pro.jpg',
   'WIN_20250310_18_20_51_Pro.jpg',
   'WIN_20250310_18_20_52_Pro.jpg',
   'WIN_20250310_18_20_53_Pro.jpg',
   'WIN_20250310_18_20_55_Pro.jpg',
   'WIN_20250310_18_20_56_Pro.jpg',
   'WIN_20250310_18_20_57_Pro.jpg',
   'WIN_20250310_

In [17]:
datas_name_emb = []
for dir, files in datas_dir_file:
    tmp = []
    for file in files:
        emb = get_face_embedding(f"./data/{dir}/{file}")
        tmp.append(emb)
    datas_name_emb.append([dir, tmp])

# datas_name_emb[0][0]

In [18]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_cvt = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    faces = app.get(frame_cvt)

    if len(faces) > 0:
        for face in faces:
            box = face.bbox.astype(np.int64)
            # cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)

            tmp_score = []
            for name, embs in datas_name_emb:
                tmp_high_score = 0
                for emb in embs:
                    similarity_score, is_same_person = compare_faces_cosine(face.embedding, emb)
                    if similarity_score > tmp_high_score:
                        tmp_high_score = similarity_score

                tmp_score.append(tmp_high_score)


            if np.max(np.array(tmp_score)) > 0.7:
                cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
                cv2.putText(
                    img=frame,
                    text=datas_name_emb[np.argmax(np.array(tmp_score))][0],
                    org=(box[0], box[1] - 10),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=0.5,
                    color=(0, 255, 0),
                    thickness=2,
                )
                cv2.putText(
                    img=frame,
                    text=f"Score: {np.max(np.array(tmp_score)):.2f}",
                    org=(box[0], box[3] + 20),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=0.5,
                    color=(0, 255, 0),
                    thickness=2,
                )

                
            if np.max(np.array(tmp_score)) <= 0.7:
                cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
                cv2.putText(
                    img=frame,
                    text="Unknown",
                    org=(box[0], box[1] - 10),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=0.5,
                    color=(0, 0, 255),
                    thickness=2,
                )
                cv2.putText(
                    img=frame,
                    text=f"Score: {np.max(np.array(tmp_score)):.2f}",
                    org=(box[0], box[3] + 20),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=0.5,
                    color=(0, 0, 255),
                    thickness=2,
                )

   
    # Display output
    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()